# LeetCode #49 — Group Anagrams: gom nhóm bằng Hash Map với "khóa chuẩn hóa"

**Mức độ:** Medium · **Chủ đề:** Array, Hash Table, String, Sorting

Skeleton trên LeetCode:

```python
class Solution:
    def groupAnagrams(self, strs: List[str]) -> List[List[str]]:
```

Mục tiêu của notebook là học một kỹ thuật cực kỳ hay dùng: **gom nhóm bằng Hash Map theo một "khóa chuẩn hóa" (canonical key)** — hai phần tử thuộc cùng nhóm **khi và chỉ khi** chúng có cùng khóa.

---

## Sau notebook này, bạn sẽ nắm được

1. Anagram là gì, và hai cách kiểm tra: **sắp xếp** và **đếm ký tự**.
2. Vì sao so sánh từng cặp chuỗi tốn `O(n²)`.
3. Ý tưởng **khóa chuẩn hóa**: `"eat"`, `"tea"`, `"ate"` đều cho khóa `"aet"`.
4. `dict` với value là list, `setdefault`, và `collections.defaultdict`.
5. Chọn khóa: chuỗi đã sắp xếp vs. **tuple đếm 26 chữ cái**.

# 1. Bài toán là gì?

Cho mảng chuỗi `strs`. Gom các **anagram** vào cùng một nhóm. Thứ tự các nhóm và thứ tự trong nhóm tùy ý.

**Anagram**: hai chuỗi có **cùng các ký tự với cùng số lần**, chỉ khác thứ tự. Ví dụ `"listen"` và `"silent"`.

**Constraints (theo đề):**

- `1 <= strs.length <= 10^4`
- `0 <= strs[i].length <= 100`
- `strs[i]` chỉ gồm chữ cái thường tiếng Anh.

---

## Ví dụ

```text
["eat", "tea", "tan", "ate", "nat", "bat"]
-> [["bat"], ["nat", "tan"], ["ate", "eat", "tea"]]

[""]  -> [[""]]
["a"] -> [["a"]]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Bảng băm (hash table) — `dict` và `set`
2. Sắp xếp (sorting)
3. Chuỗi (string) trong Python

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [1]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [2]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 260.13 ms | trên set: 0.071 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [3]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [4]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.2 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [5]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [6]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [7]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.3 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [8]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [9]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [10]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [11]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.4 Áp dụng vào bài này

Hai từ là anagram khi chúng có cùng **khóa chuẩn hóa**: chuỗi đã **sắp xếp** (`"".join(sorted(w))`) hoặc tuple đếm 26 chữ cái (`ord(c) − ord('a')`). **Nhóm theo khóa** bằng `defaultdict(list)` — đúng mẫu "nhóm" của bảng băm.

# 3. Nền tảng: kiểm tra hai chuỗi có phải anagram

## Cách 1: sắp xếp rồi so sánh

`sorted("eat")` trả về list ký tự `['a', 'e', 't']`. Hai anagram cho cùng kết quả.

## Cách 2: đếm ký tự

`Counter("eat") == Counter("tea")`.

In [12]:
from typing import List
from collections import Counter

print(sorted("eat"), sorted("tea"), sorted("tan"))
print("".join(sorted("eat")))
print(Counter("eat") == Counter("tea"), Counter("eat") == Counter("tan"))

['a', 'e', 't'] ['a', 'e', 't'] ['a', 'n', 't']
aet
True False


# 4. Cách dễ nghĩ nhất: so từng cặp

Với mỗi chuỗi, tìm nhóm đã có mà nó là anagram của phần tử đại diện; không có thì tạo nhóm mới.

In [13]:
def group_pairwise(strs: List[str]) -> List[List[str]]:
    groups = []
    for s in strs:
        for g in groups:
            if sorted(g[0]) == sorted(s):
                g.append(s)
                break
        else:                                 # for ... else: chạy khi KHÔNG có break
            groups.append([s])
    return groups

print(group_pairwise(["eat", "tea", "tan", "ate", "nat", "bat"]))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


**`for ... else`:** khối `else` của vòng `for` chạy khi vòng lặp kết thúc **mà không gặp `break`** — tức không tìm thấy nhóm phù hợp.

**Vấn đề:** mỗi chuỗi có thể phải so với **mọi** nhóm. Với `10^4` chuỗi đôi một không phải anagram, có `~5 * 10^7` lần so sánh, mỗi lần còn phải sắp xếp. Quá chậm.

# 5. Ý tưởng: khóa chuẩn hóa

Thay vì **so từng cặp**, hãy cho mỗi chuỗi một **"nhãn"** sao cho:

> Hai chuỗi là anagram **khi và chỉ khi** chúng có **cùng nhãn**.

Chuỗi đã sắp xếp là một nhãn như vậy:

| Chuỗi | Khóa (sorted) |
|---|---|
| eat | aet |
| tea | aet |
| tan | ant |
| ate | aet |
| nat | ant |
| bat | abt |

Bây giờ chỉ cần **gom theo khóa** bằng `dict`: `khóa → list các chuỗi`. Tra cứu và thêm vào `dict` là `O(1)` trung bình (đã học ở Two Sum).

In [14]:
for s in ["eat", "tea", "tan", "ate", "nat", "bat"]:
    print(f"{s} -> {''.join(sorted(s))}")

eat -> aet
tea -> aet
tan -> ant
ate -> aet
nat -> ant
bat -> abt


## `dict` có value là list

Ba cách viết "thêm `s` vào nhóm của `key`":

```python
# 1. Kiểm tra thủ công
if key not in groups:
    groups[key] = []
groups[key].append(s)

# 3. setdefault: trả về groups[key], tự tạo [] nếu chưa có
groups.setdefault(key, []).append(s)

# 4. defaultdict(list): truy cập key chưa có sẽ tự tạo []
groups = defaultdict(list)
groups[key].append(s)
```

In [15]:
from collections import defaultdict

groups = defaultdict(list)
groups["aet"].append("eat")
groups["aet"].append("tea")
groups["ant"].append("tan")
print(dict(groups))
print(list(groups.values()))

{'aet': ['eat', 'tea'], 'ant': ['tan']}
[['eat', 'tea'], ['tan']]


# 6. Lời giải chính

In [16]:
def group_anagrams(strs: List[str]) -> List[List[str]]:
    groups = defaultdict(list)
    for s in strs:
        key = "".join(sorted(s))
        groups[key].append(s)
    return list(groups.values())

print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))
print(group_anagrams([""]))
print(group_anagrams(["a"]))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]
[['']]
[['a']]


# 7. Mô phỏng từng bước

In [17]:
def trace_group_anagrams(strs: List[str]) -> List[List[str]]:
    groups = defaultdict(list)
    for s in strs:
        key = "".join(sorted(s))
        new = key not in groups
        groups[key].append(s)
        print(f"{s!r:7} khóa = {key!r:7} {'(nhóm mới)' if new else '(nhóm đã có)':14} -> {dict(groups)}")
    return list(groups.values())

_ = trace_group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])

'eat'   khóa = 'aet'   (nhóm mới)     -> {'aet': ['eat']}
'tea'   khóa = 'aet'   (nhóm đã có)   -> {'aet': ['eat', 'tea']}
'tan'   khóa = 'ant'   (nhóm mới)     -> {'aet': ['eat', 'tea'], 'ant': ['tan']}
'ate'   khóa = 'aet'   (nhóm đã có)   -> {'aet': ['eat', 'tea', 'ate'], 'ant': ['tan']}
'nat'   khóa = 'ant'   (nhóm đã có)   -> {'aet': ['eat', 'tea', 'ate'], 'ant': ['tan', 'nat']}
'bat'   khóa = 'abt'   (nhóm mới)     -> {'aet': ['eat', 'tea', 'ate'], 'ant': ['tan', 'nat'], 'abt': ['bat']}


# 8. Khóa tốt hơn: tuple đếm 26 chữ cái

Sắp xếp mỗi chuỗi tốn `O(k log k)` (`k` là độ dài). Vì chỉ có 26 chữ thường, ta có thể **đếm** trong `O(k)`:

```python
count = [0] * 26
for c in s:
    count[ord(c) - ord('a')] += 1
key = tuple(count)       # list không làm khóa dict được, tuple thì được
```

Hai anagram có cùng mảng đếm → cùng khóa.

In [18]:
def count_key(s: str) -> tuple:
    count = [0] * 26
    for c in s:
        count[ord(c) - ord("a")] += 1
    return tuple(count)

print(count_key("eat"))
print(count_key("eat") == count_key("tea"), count_key("eat") == count_key("tan"))

try:
    {[1, 2]: "x"}
except TypeError as e:
    print("list không làm khóa được:", e)

(1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0)
True False
list không làm khóa được: unhashable type: 'list'


In [19]:
def group_anagrams_count(strs: List[str]) -> List[List[str]]:
    groups = defaultdict(list)
    for s in strs:
        groups[count_key(s)].append(s)
    return list(groups.values())

print(group_anagrams_count(["eat", "tea", "tan", "ate", "nat", "bat"]))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


**Khi nào dùng khóa nào?**

- Chuỗi ngắn (≤ 100): khóa `sorted` gọn, đủ nhanh, và trong Python `sorted` chạy bằng C nên thường **nhanh hơn** vòng `for` đếm thủ công.
- Chuỗi dài, bảng chữ nhỏ: khóa đếm có độ phức tạp tốt hơn.

# 9. Phiên bản đúng format LeetCode

In [20]:
class Solution:
    def groupAnagrams(self, strs: List[str]) -> List[List[str]]:
        groups = defaultdict(list)
        for s in strs:
            groups["".join(sorted(s))].append(s)
        return list(groups.values())

Vì thứ tự tùy ý, ta chuẩn hóa kết quả trước khi so sánh: sắp xếp trong mỗi nhóm, rồi sắp xếp các nhóm.

In [21]:
def normalize(groups):
    return sorted(sorted(g) for g in groups)

def check(name, actual, expected):
    assert normalize(actual) == normalize(expected), (name, actual, expected)
    print(f"{name:<24} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.groupAnagrams(["eat", "tea", "tan", "ate", "nat", "bat"]),
      [["bat"], ["nat", "tan"], ["ate", "eat", "tea"]])
check("Ví dụ 2: chuỗi rỗng", sol.groupAnagrams([""]), [[""]])
check("Ví dụ 3", sol.groupAnagrams(["a"]), [["a"]])
check("Chuỗi lặp", sol.groupAnagrams(["ab", "ba", "ab"]), [["ab", "ab", "ba"]])
check("Khác độ dài", sol.groupAnagrams(["a", "aa", "aaa"]), [["a"], ["aa"], ["aaa"]])
check("Cùng chữ, khác số lần", sol.groupAnagrams(["aab", "abb"]), [["aab"], ["abb"]])

import random
random.seed(49)
for _ in range(1000):
    strs = ["".join(random.choice("abc") for _ in range(random.randint(0, 4))) for _ in range(random.randint(1, 12))]
    expected = normalize(group_pairwise(strs))
    assert normalize(sol.groupAnagrams(strs)) == expected
    assert normalize(group_anagrams_count(strs)) == expected
print("1000 test ngẫu nhiên: ba cách khớp nhau")

Ví dụ 1                  PASS  [['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]
Ví dụ 2: chuỗi rỗng      PASS  [['']]
Ví dụ 3                  PASS  [['a']]
Chuỗi lặp                PASS  [['ab', 'ba', 'ab']]
Khác độ dài              PASS  [['a'], ['aa'], ['aaa']]
Cùng chữ, khác số lần    PASS  [['aab'], ['abb']]


1000 test ngẫu nhiên: ba cách khớp nhau

# 10. Vì sao đúng? Độ phức tạp?

**Đúng:** khóa `sorted(s)` thỏa: `a` và `b` là anagram ⇔ `sorted(a) == sorted(b)`. Vì vậy mỗi nhóm trong `dict` chứa **đúng** các chuỗi là anagram của nhau, và các nhóm khác nhau không phải anagram của nhau.

Gọi `n = len(strs)`, `k` = độ dài chuỗi dài nhất:

```text
Khóa sorted:  Time O(n · k log k),  Space O(n · k)
Khóa đếm:     Time O(n · k),        Space O(n · k)  (mỗi khóa 26 số)
So từng cặp:  Time O(n² · k log k)
```

# 11. Những lỗi dễ mắc

## Lỗi 1: Dùng `sorted(s)` (list) làm khóa

In [22]:
try:
    d = {}
    d[sorted("eat")] = ["eat"]
except TypeError as e:
    print("Lỗi:", e)

Lỗi: unhashable type: 'list'


Phải đổi thành chuỗi (`"".join(...)`) hoặc `tuple(...)`.

## Lỗi 2: Dùng `set(s)` làm khóa

`set` bỏ mất **số lần** xuất hiện: `"aab"` và `"abb"` cùng `{a, b}` nhưng không phải anagram.

In [23]:
def wrong_set_key(strs: List[str]) -> List[List[str]]:
    groups = defaultdict(list)
    for s in strs:
        groups[frozenset(s)].append(s)            # SAI: mất thông tin số lần
    return list(groups.values())

print("Sai :", wrong_set_key(["aab", "abb", "ab"]))
print("Đúng:", group_anagrams(["aab", "abb", "ab"]))

Sai : [['aab', 'abb', 'ab']]
Đúng: [['aab'], ['abb'], ['ab']]


## Lỗi 3: Dùng tổng mã ký tự làm khóa

`sum(ord(c) for c in s)` — hai chuỗi khác nhau có thể có cùng tổng (`"ad"` và `"bc"`), gây gom nhầm.

In [24]:
print(sum(map(ord, "ad")), sum(map(ord, "bc")))

197 197


## Lỗi 4: Quên chuỗi rỗng

`""` hợp lệ, khóa của nó là `""` — vẫn hoạt động đúng với `defaultdict`. Đừng viết `if not s: continue`.

# 12. Mẹo nhận diện pattern

**Gom nhóm theo khóa chuẩn hóa** khi đề yêu cầu "nhóm các phần tử **tương đương**":

| Bài | Khóa |
|---|---|
| #49 Group Anagrams | `sorted(s)` hoặc tuple đếm |
| #249 Group Shifted Strings | hiệu giữa các ký tự liên tiếp (mod 26) |
| #609 Find Duplicate File | nội dung file |
| Gom điểm theo đường thẳng | độ dốc rút gọn `(dy/g, dx/g)` |

Câu hỏi then chốt: *"Hàm `key(x)` nào cho kết quả giống nhau **khi và chỉ khi** hai phần tử tương đương?"*

## Template

```python
groups = defaultdict(list)
for x in items:
    groups[key(x)].append(x)
return list(groups.values())
```

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Gom nhóm `["listen", "silent", "enlist", "google", "gooegl", "cat", "act", "tac", "dog"]`. Có bao nhiêu nhóm? Khóa của mỗi nhóm là gì?

## Bài 2 — Valid Anagram (LeetCode #242)

```python
is_anagram("anagram", "nagaram") -> True
is_anagram("rat", "car")         -> False
```

In [25]:
def exercise_2_is_anagram(s: str, t: str) -> bool:
    # TODO
    pass


# assert exercise_2_is_anagram("anagram", "nagaram") is True
# assert exercise_2_is_anagram("rat", "car") is False
# assert exercise_2_is_anagram("a", "aa") is False

## Bài 3 — Điền chỗ trống (dùng khóa đếm)

In [26]:
def exercise_3_group_anagrams(strs: List[str]) -> List[List[str]]:
    groups = {}
    for s in strs:
        # TODO 1: tạo count 26 phần tử, đếm ký tự
        # TODO 2: key = tuple(count)
        # TODO 3: thêm s vào groups[key] (dùng setdefault)
        pass
    return list(groups.values())


# assert normalize(exercise_3_group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])) == normalize([["bat"], ["nat", "tan"], ["ate", "eat", "tea"]])
# assert exercise_3_group_anagrams([""]) == [[""]]

## Bài 4 — Biến thể: Group Shifted Strings (LeetCode #249)

Hai chuỗi thuộc cùng nhóm nếu có thể "dịch" mọi ký tự của chuỗi này cùng một số bước (vòng tròn `z → a`) để được chuỗi kia. Ví dụ `"abc" → "bcd" → ... → "xyz"`, và `"az" → "ba"`.

```python
group_shifted(["abc", "bcd", "acef", "xyz", "az", "ba", "a", "z"])
-> [["abc", "bcd", "xyz"], ["acef"], ["az", "ba"], ["a", "z"]]
```

Gợi ý nhanh: khóa là tuple các **hiệu** `(s[i+1] - s[i]) mod 26`.

In [27]:
def exercise_4_group_shifted(strs: List[str]) -> List[List[str]]:
    # TODO
    pass


# assert normalize(exercise_4_group_shifted(["abc", "bcd", "acef", "xyz", "az", "ba", "a", "z"])) == normalize([["abc", "bcd", "xyz"], ["acef"], ["az", "ba"], ["a", "z"]])

## Bài 5 — Mini challenge: Find All Anagrams in a String (LeetCode #438)

Tìm mọi chỉ số bắt đầu của các **chuỗi con** của `s` là anagram của `p`. Yêu cầu `O(len(s))`.

```python
find_anagrams("cbaebabacd", "abc") -> [0, 6]
find_anagrams("abab", "ab")        -> [0, 1, 2]
```

Gợi ý nhanh: **cửa sổ trượt kích thước cố định** `len(p)` (bài #3) + so sánh mảng đếm 26 chữ cái.

In [28]:
def exercise_5_find_anagrams(s: str, p: str) -> List[int]:
    # TODO
    pass


# assert exercise_5_find_anagrams("cbaebabacd", "abc") == [0, 6]
# assert exercise_5_find_anagrams("abab", "ab") == [0, 1, 2]
# assert exercise_5_find_anagrams("a", "ab") == []

# 14. Gợi ý

**Bài 1**
- Mức 1: tính `"".join(sorted(w))` cho từng từ.

**Bài 2**
- Mức 1: `Counter(s) == Counter(t)` hoặc `sorted(s) == sorted(t)`.

**Bài 3**
- Mức 1: `groups.setdefault(key, []).append(s)`.

**Bài 4**
- Mức 1: `tuple((ord(s[i+1]) - ord(s[i])) % 26 for i in range(len(s) - 1))`.
- Mức 2: chuỗi 1 ký tự có khóa là tuple rỗng `()` → cùng nhóm.

**Bài 5**
- Mức 1: `need = count(p)`, `window = count(s[:len(p)])`.
- Mức 2: trượt: thêm `s[i]`, bớt `s[i - len(p)]`; so `window == need` (so 26 số là `O(1)`).

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [29]:
words = ["listen", "silent", "enlist", "google", "gooegl", "cat", "act", "tac", "dog"]
g = defaultdict(list)
for w in words:
    g["".join(sorted(w))].append(w)
for k, v in g.items():
    print(f"{k!r:10} -> {v}")
print("Số nhóm:", len(g))

'eilnst'   -> ['listen', 'silent', 'enlist']
'eggloo'   -> ['google', 'gooegl']
'act'      -> ['cat', 'act', 'tac']
'dgo'      -> ['dog']
Số nhóm: 4


## Đáp án Bài 2

In [30]:
def answer_2_is_anagram(s: str, t: str) -> bool:
    return len(s) == len(t) and Counter(s) == Counter(t)

assert answer_2_is_anagram("anagram", "nagaram") is True
assert answer_2_is_anagram("rat", "car") is False
assert answer_2_is_anagram("a", "aa") is False
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [31]:
def answer_3_group_anagrams(strs: List[str]) -> List[List[str]]:
    groups = {}
    for s in strs:
        count = [0] * 26
        for c in s:
            count[ord(c) - ord("a")] += 1
        groups.setdefault(tuple(count), []).append(s)
    return list(groups.values())

assert normalize(answer_3_group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])) == normalize([["bat"], ["nat", "tan"], ["ate", "eat", "tea"]])
assert answer_3_group_anagrams([""]) == [[""]]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [32]:
def answer_4_group_shifted(strs: List[str]) -> List[List[str]]:
    groups = defaultdict(list)
    for s in strs:
        key = tuple((ord(s[i + 1]) - ord(s[i])) % 26 for i in range(len(s) - 1))
        groups[key].append(s)
    return list(groups.values())

assert normalize(answer_4_group_shifted(["abc", "bcd", "acef", "xyz", "az", "ba", "a", "z"])) == normalize([["abc", "bcd", "xyz"], ["acef"], ["az", "ba"], ["a", "z"]])
print("Bài 4 PASS")

Bài 4 PASS


Với `"az"`: hiệu `('z' - 'a') % 26 = 25`. Với `"ba"`: `('a' - 'b') % 26 = -1 % 26 = 25`. Cùng khóa! Toán tử `%` của Python luôn trả số không âm khi số chia dương — rất tiện ở đây.

## Đáp án Bài 5

In [33]:
def answer_5_find_anagrams(s: str, p: str) -> List[int]:
    k = len(p)
    if k > len(s):
        return []
    need = [0] * 26
    window = [0] * 26
    for c in p:
        need[ord(c) - 97] += 1
    result = []
    for i, c in enumerate(s):
        window[ord(c) - 97] += 1
        if i >= k:
            window[ord(s[i - k]) - 97] -= 1
        if i >= k - 1 and window == need:
            result.append(i - k + 1)
    return result

assert answer_5_find_anagrams("cbaebabacd", "abc") == [0, 6]
assert answer_5_find_anagrams("abab", "ab") == [0, 1, 2]
assert answer_5_find_anagrams("a", "ab") == []
print("Bài 5 PASS")

Bài 5 PASS


# 16. Tổng kết

> Thay vì so từng cặp, cho mỗi phần tử một **khóa chuẩn hóa** sao cho tương đương ⇔ cùng khóa, rồi gom bằng `defaultdict(list)`. Với anagram, khóa là `"".join(sorted(s))` hoặc tuple đếm 26 chữ cái.

---

## Checklist trước khi nộp

- [ ] Khóa là chuỗi / tuple (không phải list)?
- [ ] Khóa giữ được **số lần** xuất hiện (không dùng `set`)?
- [ ] Không bỏ qua chuỗi rỗng?
- [ ] Trả `list(groups.values())`?